In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("HF_write")
secret_value_2 = user_secrets.get_secret("kaggle_key")
secret_value_3 = user_secrets.get_secret("KAGGLE_USERNAME")
secret_value_4 = user_secrets.get_secret("NGROK_AUTH_TOKEN")


In [2]:
# 1. Surgically upgrade ONLY the transformers and tokenizers
!pip install --upgrade -q transformers tokenizers accelerate bitsandbytes

# 2. Ensure torchvision stays uninstalled to prevent the Kaggle circular import bug
!pip uninstall -y -q torchvision

In [3]:
!pip install -q fastapi uvicorn pydantic pyngrok nest_asyncio

In [1]:
import os
import glob
import torch
import nest_asyncio
import uvicorn
import asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

# 1. Patch the notebook's event loop
nest_asyncio.apply()

# 2. Ngrok Authentication
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(ngrok_token)

# 3. Auto-Detect Your Attached Gemma 4 Model
print("🔍 Searching for attached Gemma 4 model in Kaggle Inputs...")
model_paths = glob.glob("/kaggle/input/**/config.json", recursive=True)

if model_paths:
    model_id = os.path.dirname(model_paths[0])
    print(f"✅ Found local model at: {model_id}")
else:
    model_id = "google/gemma-4-12b-it"

# 4. Load Tokenizer and Inject Template
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{'<start_of_turn>model\n'}}{% endif %}"
)

# 5. Load the Model
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("🔄 Mounting Gemma 4 weights to VRAM...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)
print("✅ Model successfully loaded!")

# 6. FastAPI Setup
app = FastAPI(title="Gemma 4 Native Kaggle API")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1024

@app.post("/v1/chat/completions")
async def chat_completions(request: ChatCompletionRequest):
    try:
        formatted_chat = []
        for m in request.messages:
            role = "model" if m.role == "assistant" else "user" 
            formatted_chat.append({"role": role, "content": m.content})

        prompt = tokenizer.apply_chat_template(formatted_chat, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=request.max_tokens,
                temperature=request.temperature,
                do_sample=True if request.temperature > 0.0 else False
            )
            
        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        return {
            "id": "chatcmpl-gemma4",
            "object": "chat.completion",
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": generated_text.strip()},
                "finish_reason": "stop"
            }]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 7. Open the Web Tunnel
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"🚀 API IS LIVE AT: {public_url.public_url}")
print("="*60 + "\n")

# 8. Start the Server AND KEEP THE CELL ALIVE
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
print("🟢 Uvicorn server running. This cell will now stay active!")

await server.serve()

🔍 Searching for attached Gemma 4 model in Kaggle Inputs...
✅ Found local model at: /kaggle/input/models/google/gemma-4/transformers/gemma-4-12b/2
🔄 Mounting Gemma 4 weights to VRAM...


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ Model successfully loaded!


INFO:     Started server process [427]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 API IS LIVE AT: https://countless-irrigate-vitality.ngrok-free.dev

🟢 Uvicorn server running. This cell will now stay active!
INFO:     45.80.184.36:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     45.80.184.36:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [427]


In [3]:
!pip install -q gradio

In [4]:
import os
import glob
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

# 1. Auto-Detect Your Attached Kaggle Model
print("🔍 Searching for attached Gemma 4 model in Kaggle Inputs...")
model_paths = glob.glob("/kaggle/input/**/config.json", recursive=True)

if model_paths:
    model_id = os.path.dirname(model_paths[0])
    print(f"✅ Found local model at: {model_id}")
else:
    print("⚠️ Local model not found, falling back to HuggingFace download...")
    model_id = "google/gemma-4-12b-it"

# 2. Load Tokenizer and Inject Template
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{'<start_of_turn>model\n'}}{% endif %}"
)

# 3. Load the Model across 2x T4 GPUs
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("🔄 Mounting Gemma 4 weights to VRAM...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)
print("✅ Model successfully loaded!")

# 4. Define the Chat Prediction Function
def predict(message, history):
    # Convert Gradio history format to Gemma format
    formatted_chat = []
    for user_msg, bot_msg in history:
        if user_msg:
            formatted_chat.append({"role": "user", "content": user_msg})
        if bot_msg:
            formatted_chat.append({"role": "model", "content": bot_msg})
            
    # Add the current message
    formatted_chat.append({"role": "user", "content": message})

    # Apply template and tokenize
    prompt = tokenizer.apply_chat_template(formatted_chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True
        )
        
    # Decode only the newly generated tokens
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

# 5. Launch the Chat Interface
print("🚀 Launching Gradio Web Interface...")
demo = gr.ChatInterface(
    fn=predict,
    title="Gemma 4 Live Testing Ground",
    description="Running directly on Kaggle Dual T4 GPUs.",
    theme="soft"
)

# share=True creates a public link you can access from your local computer
demo.launch(share=True, inline=False)

🔍 Searching for attached Gemma 4 model in Kaggle Inputs...
✅ Found local model at: /kaggle/input/models/google/gemma-4/transformers/gemma-4-12b/2
🔄 Mounting Gemma 4 weights to VRAM...


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ Model successfully loaded!
🚀 Launching Gradio Web Interface...


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ffa0f7ffa96f2a08c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
